# Gold (B): reference and external data

**Owner:** Person B  
**Builds:** `dim_products`, `dim_sellers`, `dim_date`, `dim_geography`, `fact_reviews`, `mart_seller_performance`, `mart_daily_sales_weather`

## 1. `gold.dim_products`
- **Grain:** one row per `product_id`.
- **Columns:** `product_id`, `category_name` (Portuguese), `category_name_english`, `weight_g`, `length_cm`, `height_cm`, `width_cm`, `photos_qty`.
- **Logic:** left join `silver.category_translation`; use `COALESCE(category_name_english, category_name, 'unknown')` so untranslated or missing categories do not disappear.
- **Checks:** `product_id` unique; row count equals `silver.products`.

## 2. `gold.dim_sellers`
- **Grain:** one row per `seller_id`.
- **Columns:** `seller_id`, `zip_code_prefix`, `city`, `state`.
- **Checks:** `seller_id` unique; row count equals `silver.sellers`.

## 3. `gold.dim_date`
- **Grain:** one row per day, continuous with no gaps, from 2016-01-01 to 2018-12-31.
- **Columns:** `date_key` (DATE), `year`, `quarter`, `month`, `month_name`, `week_of_year`, `day_of_week`, `is_weekend`, `is_holiday`, `holiday_name`, `usd_per_brl`.
- **Logic:** generate the calendar with a recursive CTE or a numbers table, left join holidays and exchange rates. For days without a rate (weekends), use the last known rate; document this rule.
- **Power BI:** mark this table as the date table.
- **Dependency:** reads `silver.holidays` (built by Person A) and `silver.exchange_rates`.

## 4. `gold.dim_geography`
- **Grain:** one row per `zip_code_prefix`.
- **Columns:** `zip_code_prefix`, `city`, `state`, `lat` (average), `lng` (average).
- **Logic:** silver geolocation has many rows per zip prefix; take the average coordinates per zip. For city and state use the most frequent value per zip. This averaging is a simplification, state it in the log.
- **Use:** map visuals in Power BI.

## 5. `gold.fact_reviews`
- **Grain:** one row per `order_id` (reviews aggregated, because some orders have several reviews).
- **Columns:** `order_id`, `review_score_avg`, `review_count`, `first_review_date`, `has_comment` (BIT).
- **Note:** do not store the comment text here; it makes the Power BI model heavy.
- **Checks:** `order_id` unique; every `order_id` exists in `fact_orders` (report orphans in the log).

## 6. `gold.mart_seller_performance`
- **Grain:** one row per `seller_id`.
- **Columns:** `seller_id`, `state`, `orders_count`, `items_sold`, `revenue`, `avg_review_score`, `late_orders_pct`, `avg_delivery_days`.
- **Logic:** join `fact_order_items`, `fact_orders` and `fact_reviews` at order level, aggregate per seller, then join `dim_sellers`.
- **Dependency:** needs Person A's `fact_orders` and `fact_order_items`.

## 7. `gold.mart_daily_sales_weather`
- **Grain:** one row per day, **São Paulo customers only** (state `SP`).
- **Columns:** `date_key`, `orders_count`, `revenue`, `temp_max`, `temp_min`, `precipitation`, `is_holiday`, `is_rainy` (BIT).
- **Important limitation:** the weather data covers São Paulo only. Do not join it to orders from other states. Put "SP only" in the table log and in any Power BI visual title that uses it.
- **Dependency:** needs Person A's `fact_orders` and `dim_customers`.